In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import os
from google.colab import userdata

# 1. SETUP ENVIRONMENT & AUTH
hf_token = userdata.get('HF_TOKEN')
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" # Fast Rust-based downloader
model_path = "./my_local_model"

# 2. INSTALL FAST TRANSFER TOOL
!pip install -q hf_transfer

# 3. DOWNLOAD THE MODEL (The Failsafe Part)
print("Downloading model via CLI (Fast Transfer enabled)...")
!huggingface-cli download Jaiccc/model_0_streaming_timestamp \
    --local-dir {model_path} \
    --token {hf_token}

# 4. LOAD THE MODEL FROM DISK
from unsloth import FastLanguageModel
import torch

# Ensure ModelScope isn't interfering
if "UNSLOTH_USE_MODELSCOPE" in os.environ:
    del os.environ["UNSLOTH_USE_MODELSCOPE"]

max_seq_length = 4096
load_in_4bit = True

print("\nLoading model from local storage...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path, # Path to the folder we just downloaded
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)
print("Model loaded successfully from local disk!")

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
.gitattributes: 1.52kB [00:00, 1.33MB/s]
Download complete. Moving file to my_local_model/.gitattributes
README.md: 3.25kB [00:00, 15.2MB/s]
Download complete. Moving file to my_local_model/README.md
adapter_config.json: 1.20kB [00:00, 7.80MB/s]
Download complete. Moving file to my_local_model/adapter_config.json
adapter_model.safetensors: 100% 262M/262M [00:03<00:00, 78.8MB/s]
Download complete. Moving file to my_local_model/adapter_model.safetensors
chat_template.jinja: 100% 462/462 [00:00<00:00, 4.87MB/s]
Download complete. Moving file to my_local_model/chat_template.jinja
merges.txt: 917kB [00:00, 119MB/s]
Download complete. Moving file to my_local_model/merges.txt
special_tokens_map.json: 100% 570/570 [00:00<00:00, 6.21MB/s]
Download complete. Moving file to my_local_model/special_tokens_map.json
tokenizer.json: 7.15MB [00:00, 250MB/s]
Download complete. Moving file to my_local_model/tokenizer.json
t

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/170 [00:00<?, ?B/s]

Unsloth: Will load ./my_local_model as a legacy tokenizer.
Unsloth 2026.3.10 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


Model loaded successfully from local disk!


In [ ]:
# System prompt identical to training
SYSTEM_PROMPT = """Your task is to analyze terminal XML logs and determine whether the timestamp in the TARGET LINE belongs to a "new event" or an "old event".

### DEFINITION OF A NEW EVENT:
1. **Explicit Prompts:** The very first `<user_input>` that immediately follows a shell prompt (e.g., `demo@faiserver:~$`).
2. **Phase Transitions:** In automated logs, moving from one major build stage to another (e.g., from 'fai-mirror finished' to 'Copying the nfsroot').
3. **Internal Logic:** Shifts from downloading to processing.

### WHAT IS *NOT* A NEW EVENT (OLD EVENT):
- **User Input / Keystrokes:** A user typing a command, including pressing the Enter key (a newline `\\n`), is just the completion of the input phase.
- **Incomplete Tasks:** Continuous system output without a clear phase shift.

CRITICAL INSTRUCTION: You must classify ONLY the timestamp found in the "### TARGET LINE" section. Do NOT extract timestamps from the "### CONTEXT" section. Output only the timestamp and the classification. Do NOT use brackets, periods, explanations, or markdown formatting.

Output Format Example 1: 39.229814, old event
Output Format Example 2: 111.602501, new event"""


# Regex compilers
CHUNK_RE = re.compile(r"<(user_input|system_output)\b[^>]*?(?:/>|>.*?</\1>)", flags=re.DOTALL)
TIME_RE = re.compile(r'timestamp="([\d\.]+)"')

def get_timestamp(chunk_text: str) -> str:
    match = TIME_RE.search(chunk_text)
    return match.group(1) if match else ""

def truncate_single_chunk(raw_text: str, tag_type: str, max_lines: int = 15) -> str:
    """Compresses massive system outputs to save context length (matches training logic)."""
    if tag_type != "system_output" or raw_text.endswith("/>"):
        return raw_text

    first_close = raw_text.find(">")
    last_open = raw_text.rfind("</")

    if first_close == -1 or last_open == -1: return raw_text

    opening_tag = raw_text[:first_close+1]
    closing_tag = raw_text[last_open:]
    inner_text = raw_text[first_close+1:last_open]

    lines = inner_text.split('\n')
    if len(lines) > max_lines:
        head = '\n'.join(lines[:5])
        tail = '\n'.join(lines[-5:])
        removed = len(lines) - 10
        return f"{opening_tag}{head}\n\n... [TRUNCATED {removed} LINES] ...\n\n{tail}{closing_tag}"

    return raw_text

In [ ]:
import re
from google.colab import files
from tqdm.notebook import tqdm

# ==========================================
# 1. SETUP & HELPER FUNCTIONS
# ==========================================
SYSTEM_PROMPT = """Your task is to analyze terminal XML logs and determine whether the timestamp in the TARGET LINE belongs to a "new event" or an "old event".

### DEFINITION OF A NEW EVENT:
1. **Explicit Prompts:** The very first `<user_input>` that immediately follows a shell prompt (e.g., `demo@faiserver:~$`).
2. **Phase Transitions:** In automated logs, moving from one major build stage to another (e.g., from 'fai-mirror finished' to 'Copying the nfsroot').
3. **Internal Logic:** Shifts from downloading to processing.

### WHAT IS *NOT* A NEW EVENT (OLD EVENT):
- **User Input / Keystrokes:** A user typing a command, including pressing the Enter key (a newline `\n`), is just the completion of the input phase.
- **Incomplete Tasks:** Continuous system output without a clear phase shift.

CRITICAL INSTRUCTION: You must classify ONLY the timestamp found in the "### TARGET LINE" section. Do NOT extract timestamps from the "### CONTEXT" section. Output only the timestamp and the classification. Do NOT use brackets, periods, explanations, or markdown formatting."""

CHUNK_RE = re.compile(r"<(user_input|system_output)\b[^>]*?(?:/>|>.*?</\1>)", flags=re.DOTALL)
TIME_RE = re.compile(r'timestamp="([\d\.]+)"')

def get_timestamp(chunk_text: str) -> str:
    match = TIME_RE.search(chunk_text)
    return match.group(1) if match else ""

def truncate_single_chunk(raw_text: str, tag_type: str, max_lines: int = 15) -> str:
    if tag_type != "system_output" or raw_text.endswith("/>"):
        return raw_text
    first_close = raw_text.find(">")
    last_open = raw_text.rfind("</")
    if first_close == -1 or last_open == -1: return raw_text

    opening_tag = raw_text[:first_close+1]
    closing_tag = raw_text[last_open:]
    inner_text = raw_text[first_close+1:last_open]

    lines = inner_text.split('\n')
    if len(lines) > max_lines:
        head = '\n'.join(lines[:5])
        tail = '\n'.join(lines[-5:])
        removed = len(lines) - 10
        return f"{opening_tag}{head}\n\n... [TRUNCATED {removed} LINES] ...\n\n{tail}{closing_tag}"
    return raw_text

def compress_context_window(chunks, max_total_lines=25):
    total_lines = sum(len(c["text"].split('\n')) for c in chunks)
    if total_lines <= max_total_lines:
        return "\n".join([c["text"] for c in chunks])

    result = []
    for idx, c in enumerate(chunks):
        raw_text = c["text"]
        if idx < 5 or idx >= len(chunks) - 5:
            result.append(raw_text)
        else:
            if "<system_output" in raw_text and not raw_text.endswith("/>"):
                first_close = raw_text.find(">")
                last_open = raw_text.rfind("</")
                if first_close != -1 and last_open != -1:
                    opening_tag = raw_text[:first_close+1]
                    closing_tag = raw_text[last_open:]
                    result.append(f"{opening_tag}... [TRUNCATED TO SAVE SPACE] ...{closing_tag}")
                else:
                    result.append(raw_text)
            else:
                result.append(raw_text)
    return "\n".join(result)

# ==========================================
# 2. UPLOAD & INFERENCE LOOP
# ==========================================
print("Please upload a raw XML terminal log:")
uploaded = files.upload()

for filename, content in uploaded.items():
    print(f"\nProcessing {filename}...")
    xml_content = content.decode('utf-8')

    # Parse and apply Phase 1 truncation
    all_chunks = []
    for match in CHUNK_RE.finditer(xml_content):
        tag_type = match.group(1)
        raw_text = match.group(0)

        # PRE-PHASE TRUNCATION: Protect against massive single-line strings bypassing the newline limit
        if len(raw_text) > 4000:
            raw_text = raw_text[:2000] + "\n...[MASSIVE LINE TRUNCATED]...\n" + raw_text[-2000:]

        processed_text = truncate_single_chunk(raw_text, tag_type, max_lines=15)
        ts_str = get_timestamp(processed_text)
        if ts_str:
            all_chunks.append({"ts": ts_str, "text": processed_text})

    WINDOW_SIZE = 15

    if len(all_chunks) == 0:
        print("No valid timestamps found.")
        continue

    print(f"Starting inference on ALL {len(all_chunks)} events...\n")
    print("-" * 50)

    # REQUIREMENT 2: Loop starts at 0 to predict EVERY single timestamp
    for i in range(len(all_chunks)):

        # Dynamically size the window for the first few events
        start_idx = max(0, i - WINDOW_SIZE + 1)
        window_chunks = all_chunks[start_idx : i + 1]

        context_chunks = window_chunks[:-1]
        target_chunk = window_chunks[-1]
        target_ts = target_chunk["ts"]

        # Format Context
        if len(context_chunks) > 0:
            context_text = compress_context_window(context_chunks, max_total_lines=25)
        else:
            context_text = "No previous context available."

        target_text = target_chunk["text"]

        # ==========================================
        # PHASE 3 TRUNCATION: ABSOLUTE CHARACTER LIMITS
        # (Fixes the Out-of-Boundary token limit crashes)
        # ==========================================
        if len(context_text) > 8000:
            context_text = "... [EARLIER CONTEXT TRUNCATED] ...\n" + context_text[-8000:]

        if len(target_text) > 2000:
            target_text = target_text[:1000] + "\n... [TARGET MIDDLE TRUNCATED] ...\n" + target_text[-1000:]

        input_data = f"### CONTEXT (Previous Events):\n{context_text}\n\n### TARGET LINE (Extract and Classify THIS Timestamp):\n{target_text}"
        combined_user_text = f"{SYSTEM_PROMPT}\n\n{input_data}"

        prompt = f"<|im_start|>user<|im_sep|>{combined_user_text}<|im_end|><|im_start|>assistant<|im_sep|>"

        # Tokenize
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        # ==========================================
        # EMERGENCY TOKEN SLICE (The Final Safety Net)
        # ==========================================
        MAX_TOKENS = 4000
        if inputs.input_ids.shape[1] > MAX_TOKENS:
            inputs.input_ids = inputs.input_ids[:, -MAX_TOKENS:]
            inputs.attention_mask = inputs.attention_mask[:, -MAX_TOKENS:]

        # Generate
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=32,
            use_cache=True,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id # Cleans up any leftover padding warnings
        )

        raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

        # Extract prediction
        m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
        model_result = m.group(1).strip().lower() if m else raw_output.split("assistant")[-1].strip().lower()

        # REQUIREMENT 1: Output directly to terminal
        icon = "🔥 NEW EVENT" if "new" in model_result else "⚪ old event"
        print(f"[{i+1:03d}/{len(all_chunks):03d}] TS: {target_ts:<10} | {icon} | Output: '{model_result}'")

    print("\n✅ Finished processing all timestamps in the file!")

Please upload a raw XML terminal log:


Saving renee_rec1.cast.xml to renee_rec1.cast.xml

Processing renee_rec1.cast.xml...
Starting inference on ALL 501 events...

--------------------------------------------------
[001/501] TS: 0.020188   | 🔥 NEW EVENT | Output: '<|im_sep|> 0.0.20188, new event<|im_end|>'
[002/501] TS: 1.002062   | 🔥 NEW EVENT | Output: '<|im_sep|> 1.00.20.62, new event<|im_end|>'
[003/501] TS: 1.002324   | ⚪ old event | Output: '<|im_sep|> 1.002324, old event<|im_end|>'
[004/501] TS: 1.14847    | ⚪ old event | Output: '<|im_sep|> 1.14847 old event<|im_end|>'
[005/501] TS: 1.148716   | ⚪ old event | Output: '<|im_sep|> 1.148716 old event<|im_end|>'
[006/501] TS: 1.240533   | ⚪ old event | Output: '<|im_sep|> 1.24.05.33, old event<|im_end|>'
[007/501] TS: 1.240762   | ⚪ old event | Output: '<|im_sep|> 1.24.07.62 old event<|im_end|>'
[008/501] TS: 1.500879   | ⚪ old event | Output: '<|im_sep|> 1.500879 old event<|im_end|>'
[009/501] TS: 1.501132   | ⚪ old event | Output: '<|im_sep|> 1.50.11.32 old event<|im